# GPS + LGBM OD-Flow Experiments

Takes pre-trained GPS models (from `gps_od.ipynb`) and trains a LightGBM decoder
on top of the frozen GPS encoder embeddings via `train_lgbm_from_model`.

**Workflow:**
1. Run `gps_od.ipynb` to train and save GPS weights
2. Run this notebook to extract embeddings and train LGBM head
3. Compare GPS-only vs GPS+LGBM results


In [ ]:
import sys, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, str(Path('.').resolve()))

from models.GPS.config import (
    device, cleanup_gpu,
    WEIGHTS_DIR, SINGLE_CITY_ID,
    TrainingConfig, load_model_config,
)
from models.GPS.data_load import prepare_single_city_data
from models.GPS.model import make_model
from models.GPS.lgbm_pipeline import train_lgbm_from_model
from models.GPS.metrics import cal_od_metrics, average_listed_metrics

print(f'Device: {device}')
print(f'Weights dir: {WEIGHTS_DIR}')


## Configuration

Select which GPS run_ids to use as encoder donors for LGBM training.
Run `gps_od.ipynb` first to generate weights for the desired configs.


In [ ]:
# GPS run_ids whose encoder will be used as LGBM feature extractor.
# These must have been trained in gps_od.ipynb (weights + JSON saved).
LGBM_DONOR_IDS = [
    'SC_TF_CE',
    'SC_TF_CE_lape',
    'SC_TF_CE_rle',
    'SC_TF_focal',
    'SC_TF_focal_lape_rle',
    'SC_BL_CE',
    'SC_BL_CE_lape',
    'SC_BL_focal',
]

trained = [d for d in LGBM_DONOR_IDS if (WEIGHTS_DIR / f'{d}.pt').exists()]
print(f'Donor run_ids: {len(LGBM_DONOR_IDS)}')
print(f'Available (weights found): {len(trained)}')
if len(trained) < len(LGBM_DONOR_IDS):
    missing = [d for d in LGBM_DONOR_IDS if d not in trained]
    print(f'Missing (run gps_od.ipynb first): {missing}')


## Data Loading

Load city data (cached per pe_type — same pattern as gps_od.ipynb).


In [ ]:
_data_cache = {}

def get_city_data(pe_type='rwpe'):
    if pe_type not in _data_cache:
        print(f'  Loading single-city data (pe_type={pe_type})...')
        _data_cache[pe_type] = prepare_single_city_data(
            area_id=SINGLE_CITY_ID, pe_type=pe_type)
        N = _data_cache[pe_type]['num_nodes']
        print(f'  N={N}')
    return _data_cache[pe_type]


## LGBM Training Loop

For each donor GPS model: load weights → extract embeddings → train LGBM.


In [ ]:
lgbm_results = {}      # lgbm_run_id -> result dict from train_lgbm_from_model
gps_results_cache = {} # donor_id -> GPS metrics (loaded alongside for comparison)

for donor_id in LGBM_DONOR_IDS:
    weight_path = WEIGHTS_DIR / f'{donor_id}.pt'
    saved_cfg   = load_model_config(donor_id)

    if not weight_path.exists():
        print(f'  [SKIP] {donor_id}: weights not found — run gps_od.ipynb first')
        continue
    if saved_cfg is None:
        print(f'  [SKIP] {donor_id}: no saved config JSON')
        continue

    city_data = get_city_data(saved_cfg.pe_type)

    # Load GPS model
    gps_model = make_model(saved_cfg, graph_data_ref=city_data['graph_data'])
    gps_model.load_state_dict(torch.load(str(weight_path), map_location=device))
    gps_model.to(device).eval()

    # GPS-only metrics (for comparison)
    from models.GPS.metrics import predict_full_matrix
    with torch.no_grad():
        pred_gps = predict_full_matrix(gps_model, city_data, saved_cfg)
    pred_gps[pred_gps < 0] = 0
    gps_results_cache[donor_id] = cal_od_metrics(pred_gps, city_data['od_matrix_np'])
    print(f'  GPS   {donor_id}: CPC={gps_results_cache[donor_id]["CPC"]:.4f}')

    # Train LGBM on GPS embeddings
    lgbm_run_id = f'{donor_id}_lgbm'
    try:
        result = train_lgbm_from_model(lgbm_run_id, city_data, gps_model, donor_id)
        lgbm_results[lgbm_run_id] = result
    except Exception as e:
        print(f'  ERROR {donor_id}: {e}')
    finally:
        del gps_model; cleanup_gpu()

print(f'\nCompleted: {len(lgbm_results)} LGBM models')


## Results Comparison

GPS-encoder-only vs GPS-encoder + LGBM-decoder.


In [ ]:
rows = {}

for lgbm_id, r in lgbm_results.items():
    donor_id = lgbm_id.replace('_lgbm', '')
    gps_m    = gps_results_cache.get(donor_id, {})
    lgbm_m   = r['metrics_full']

    rows[f'{donor_id}'] = {
        'model':    'GPS (neural)',
        'CPC':      gps_m.get('CPC',  float('nan')),
        'MAE':      gps_m.get('MAE',  float('nan')),
        'RMSE':     gps_m.get('RMSE', float('nan')),
    }
    rows[f'{donor_id}_lgbm'] = {
        'model':    'GPS emb + LGBM',
        'CPC':      lgbm_m.get('CPC',  float('nan')),
        'MAE':      lgbm_m.get('MAE',  float('nan')),
        'RMSE':     lgbm_m.get('RMSE', float('nan')),
    }

if rows:
    df = pd.DataFrame(rows).T
    df = df.sort_values('CPC', ascending=False)
    print('\n' + '=' * 80)
    print('  GPS vs GPS+LGBM (sorted by CPC)')
    print('=' * 80)
    print(df.to_string())

    import os; os.makedirs('results', exist_ok=True)
    df.to_csv('results/lgbm_comparison.csv')
    print('\nSaved to results/lgbm_comparison.csv')
else:
    print('No results to display.')


## Training Curves

LGBM iteration loss curves.


In [ ]:
# LightGBM doesn't have iteration curves by default; show bar chart instead
if rows:
    df_plot = pd.DataFrame(rows).T
    gps_rows  = df_plot[df_plot['model'] == 'GPS (neural)']
    lgbm_rows = df_plot[df_plot['model'] == 'GPS emb + LGBM']

    x = range(len(gps_rows))
    labels = [idx.replace('SC_', '') for idx in gps_rows.index]
    w = 0.35

    fig, ax = plt.subplots(figsize=(max(8, len(x)*1.5), 5))
    ax.bar([i - w/2 for i in x], gps_rows['CPC'].astype(float),
           width=w, label='GPS (neural)', color='#2196F3')
    ax.bar([i + w/2 for i in x], lgbm_rows['CPC'].astype(float),
           width=w, label='GPS emb + LGBM', color='#FF9800')
    ax.set_xticks(list(x)); ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel('CPC'); ax.set_title('GPS vs GPS+LGBM — CPC comparison')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()
